# Resume Screening 

In [1]:
import pandas as pd
import os

# Load datasets
resume_df = pd.read_csv('Dataset/Resume.csv')
jobs_df = pd.read_csv('Dataset/job_title_des.csv')

print("✅ Successfully loaded both datasets!")

print("\n--- First 5 Resumes ---")
print(resume_df.head())

print("\n--- First 5 Job Descriptions ---")
print(jobs_df.head())

✅ Successfully loaded both datasets!

--- First 5 Resumes ---
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  

--- First 5 Job Descriptions ---
   Unnamed: 0             Job Title  \
0           0     Flutter Developer   
1           1      Django Developer   
2     

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading the Sentence Transformer model... (This may take a moment on the first run)")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded successfully.")

resume_texts = resume_df['Resume_str'].fillna('').tolist()
job_texts = jobs_df['Job Description'].fillna('').tolist()

# 3. Generate the embeddings for resumes
print("\nGenerating embeddings for all resumes...")
resume_embeddings = model.encode(resume_texts, show_progress_bar=True)
print(f"✅ Generated embeddings for {len(resume_embeddings)} resumes.")


# 4. Generate the embeddings for job descriptions
print("\nGenerating embeddings for all job descriptions...")
job_embeddings = model.encode(job_texts, show_progress_bar=True)
print(f"✅ Generated embeddings for {len(job_embeddings)} job descriptions.")


# 5. Save the embeddings to files
np.save('resume_embeddings.npy', resume_embeddings)
np.save('job_embeddings.npy', job_embeddings)
print("\n✅ Embeddings saved to 'resume_embeddings.npy' and 'job_embeddings.npy'")

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the Sentence Transformer model... (This may take a moment on the first run)
✅ Model loaded successfully.

Generating embeddings for all resumes...


Batches: 100%|██████████| 78/78 [03:05<00:00,  2.37s/it]


✅ Generated embeddings for 2484 resumes.

Generating embeddings for all job descriptions...


Batches: 100%|██████████| 72/72 [02:30<00:00,  2.09s/it]

✅ Generated embeddings for 2277 job descriptions.

✅ Embeddings saved to 'resume_embeddings.npy' and 'job_embeddings.npy'


In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ---- Step 5: Calculate Cosine Similarity ----

# 1. Load the embeddings we saved earlier
# This is much faster than re-generating them!
print("Loading pre-computed embeddings...")
try:
    resume_embeddings = np.load('resume_embeddings.npy')
    job_embeddings = np.load('job_embeddings.npy')
    print("✅ Embeddings loaded successfully.")
except FileNotFoundError:
    print("❌ Error: Could not find .npy files. Please re-run Step 4 to generate embeddings.")
    # You might want to stop the script here if the files are missing
    # For now, we'll assume they exist.

# 2. Calculate the cosine similarity matrix
# This compares every resume embedding with every job embedding.
print("\nCalculating cosine similarity matrix...")
# The result is a matrix where matrix[i, j] is the similarity
# between the i-th resume and the j-th job.
cosine_sim_matrix = cosine_similarity(resume_embeddings, job_embeddings)
print(f"✅ Similarity matrix created with shape: {cosine_sim_matrix.shape}")


# ---- Step 6: Rank and Present Results ----

# Let's find the best job matches for a specific resume.
# We'll use the first resume in our dataset as an example (index 0).
resume_index = 0

# 1. Get the similarity scores for our chosen resume against all jobs.
resume_scores = cosine_sim_matrix[resume_index]

# 2. Get the indices of the top N job matches.
# We'll use np.argsort to get the indices that would sort the array,
# then we take the last 5 and reverse them to get the top 5.
top_n = 5
top_job_indices = np.argsort(resume_scores)[-top_n:][::-1]

# 3. Display the results
print(f"\n--- Top {top_n} Job Matches for Resume #{resume_index} ---")

# Get the text of the resume we are matching
# Using .iloc[resume_index] to get the specific resume's text
resume_text = resume_df['Resume_str'].iloc[resume_index]
print(f"\nOriginal Resume Category: {resume_df['Category'].iloc[resume_index]}")
# Optional: Print a snippet of the resume
# print(f"Resume Snippet: {resume_text[:200]}...\n")


for index in top_job_indices:
    # Get the job title and description from the original jobs dataframe
    job_title = jobs_df['Job Title'].iloc[index]
    
    # Get the similarity score and convert it to a percentage
    match_score = resume_scores[index] * 100
    
    print(f"✅ Match: {match_score:.2f}% | Job Title: {job_title}")

Loading pre-computed embeddings...
✅ Embeddings loaded successfully.

Calculating cosine similarity matrix...
✅ Similarity matrix created with shape: (2484, 2277)

--- Top 5 Job Matches for Resume #0 ---

Original Resume Category: HR
✅ Match: 64.66% | Job Title: Database Administrator
✅ Match: 64.42% | Job Title: Database Administrator
✅ Match: 57.51% | Job Title: Database Administrator
✅ Match: 56.66% | Job Title: Database Administrator
✅ Match: 56.34% | Job Title: Database Administrator
